In [0]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [0]:
df = spark.table('flight_delay_predictor_project.default.first_prediction_try_1').toPandas()

In [0]:

df = spark.createDataFrame(df)

In [0]:
from pyspark.sql import Window, functions as F

hr_delay = (df.withColumn("delay_in_hr", 
F.when(F.col("weather_delay") < 1, "no delay")
.when(F.col("weather_delay") < 60, "less than 1") 
.when(F.col("weather_delay") < 120, "1+")
.when(F.col("weather_delay") < 180, "2+")
.when(F.col("weather_delay") < 240, "3+")
.when(F.col("weather_delay") < 300, "4+")
.when(F.col("weather_delay") < 360, "5+")
.when(F.col("weather_delay") < 420, "6+")
.when(F.col("weather_delay") < 480, "7+")
.when(F.col("weather_delay") < 540, "8+")
.when(F.col("weather_delay") < 600, "9+")
.otherwise("10+")) 
.select("temperature_2m", "relative_humidity_2m", "precipitation", "snowfall", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "cloud_cover", "pressure_msl", "weather_code", "delay_in_hr"))

In [0]:

hr_delay = hr_delay.toPandas()
print(hr_delay)

In [0]:
flights_1 = hr_delay[hr_delay['delay_in_hr'] != 'no delay']
flights_2 = hr_delay[hr_delay['delay_in_hr'] == 'no delay']
print(flights_1.shape)
print(flights_2.shape)

In [0]:

flights_1_upsampled = resample(flights_1,random_state=42,n_samples=101915,replace=True)
flights_upsampled = pd.concat([flights_1_upsampled,flights_2])

In [0]:
X = flights_upsampled.drop('delay_in_hr',axis=1)
y = flights_upsampled['delay_in_hr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,random_state=42)
dtree = DecisionTreeClassifier()

In [0]:
dtree.fit(X_train,y_train)
     

In [0]:
from sklearn.metrics import accuracy_score

# This line instantiates the model. 
rf = RandomForestClassifier() 

# Fit the model on your training data.
rf.fit(X_train, y_train) 
y_predict = rf.predict(X_test)

# And score it on your testing data.
accuracy_score(y_test, y_predict)

In [0]:
feature_importances = pd.DataFrame(rf.feature_importances_,
                                   index = X_train.columns,
                                   columns=['importance']).sort_values('importance', ascending=False)
feature_importances